# Transformers, what can they do?

Install the Transformers, Datasets, and Evaluate libraries to run this notebook.  
Install pytorch

In [ ]:
%pip install datasets evaluate "transformers[sentencepiece]"
%pip install torch
%pip install -U huggingface_hub
%pip install ipywidgets

In [1]:
import torch, transformers
from transformers import pipeline

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)

PyTorch: 2.12.1
Transformers: 5.12.1


In [ ]:
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(device)   

In [ ]:
## pipeline() will be extremely slow without hf_token
import os
# os.environ["HF_TOKEN"] = "<REDACTED_HF_TOKEN>"

Some old NLP tasks using conventional models

In [ ]:
classifier = pipeline("sentiment-analysis")
classifier(
    ["I've been waiting for a HuggingFace course my whole life.", "I hate this so much!"]
)

In [ ]:
classifier = pipeline("zero-shot-classification")
classifier(
    "This is a course about the Transformers library",
    candidate_labels=["education", "politics", "business"],
)

In [ ]:
unmasker = pipeline("fill-mask")
unmasker("This course will teach you all about <mask> models.", top_k=2)

Since hugging-face transformers V5, it is suggested that all text-generation tasks should use LLMs

In [ ]:
# download a model from the Hugging Face Hub, to avoid network issues.
from huggingface_hub import snapshot_download

model_path = snapshot_download(
    repo_id="Qwen/Qwen2.5-0.5B-Instruct",
    cache_dir="./model_cache"
)

print("Downloaded to:", model_path)

In [ ]:
import time

dtype = torch.float16 if device == "mps" else torch.float32

print("Using:", device)
print("Loading model...")
start = time.time()

qa = pipeline(
    "text-generation",
    model=model_path,
    device=device,
    dtype=dtype,
)

print(f"Loaded in {time.time() - start:.1f} seconds")

In [ ]:

context = """
The Hugging Face Transformers library provides pretrained models for
natural-language processing, computer vision, and audio. The pipeline
function offers a simple interface for running model inference.
"""

question = "What does the pipeline function provide?"

messages = [
    {
        "role": "system",
        "content": (
            "Answer the question using only the supplied context. "
            "If the answer is not present, say: I don't know."
        ),
    },
    {
        "role": "user",
        "content": f"Context:\n{context}\n\nQuestion: {question}",
    },
]

result = qa(
    messages,
    max_new_tokens=50,
    do_sample=False,
)

answer = result[0]["generated_text"][-1]["content"]
print(answer)

In [ ]:

context = """
My name is Sylvain and I work at Hugging Face in Brooklyn."""

questions = [
    "what is my name?",
    "where am I working?",
    "which company do I work for?"
]

messages = [
    {
        "role": "system",
        "content": (
            "Answer the questions one by one,using only the supplied context. "
            "If the answer is not present, say: I don't know."
        ),
    },
    {
        "role": "user",
        "content": f"Context:\n{context}\n\nQuestions: {questions}",
    },
]

result = qa(
    messages,
    max_new_tokens=50,
    do_sample=False,
)

answer = result[0]["generated_text"][-1]["content"]
print(answer)

MultiModal LLMs for multilti-modal tasks.  
I tried google/Gemma, microsoft/Phi, but can't make them work for mac locally.  
Qwen seems working fine though.

In [2]:
import os
# os.environ["HF_TOKEN"] = "<REDACTED_HF_TOKEN>"
os.environ["HF_TOKEN"] = "<REDACTED_HF_TOKEN>"

In [3]:
import torch
from huggingface_hub import snapshot_download
from transformers import pipeline

model_id = "Qwen/Qwen2.5-VL-3B-Instruct"
cache_dir = "./model_cache"

# 1. Download model to local cache first
vl_model_path = snapshot_download(
    repo_id=model_id,
    cache_dir=cache_dir,
)

print("Downloaded model path:", vl_model_path)

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

Downloaded model path: ./model_cache/models--Qwen--Qwen2.5-VL-3B-Instruct/snapshots/66285546d2b821cf421d4f5eb2576359d3770cd3


In [4]:
# 2. Load from local path
device = "mps" if torch.backends.mps.is_available() else "cpu"

vision_qa = pipeline(
    task="image-text-to-text",
    model=vl_model_path,
    device=device,
    dtype=torch.float16 if device == "mps" else torch.float32,
)

# 3. Ask about an image
image_url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/pipeline-cat-chonk.jpeg"

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": image_url},
            {"type": "text", "text": "What is the main object in this image? Answer with one word."},
        ],
    }
]

result = vision_qa(
    text=messages,
    max_new_tokens=30,
    do_sample=False,
)

answer = result[0]["generated_text"][-1]["content"]
print(answer)

[ERROR] `loss` is part of Qwen2_5_VLCausalLMOutputWithPast.__init__'s signature, but not documented. Make sure to add it to the docstring of the function in /Users/auser/src/hf-llm-course/.env/lib/python3.14/site-packages/transformers/models/qwen2_5_vl/modeling_qwen2_5_vl.py.
[ERROR] `logits` is part of Qwen2_5_VLCausalLMOutputWithPast.__init__'s signature, but not documented. Make sure to add it to the docstring of the function in /Users/auser/src/hf-llm-course/.env/lib/python3.14/site-packages/transformers/models/qwen2_5_vl/modeling_qwen2_5_vl.py.


Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Keyword argument `do_sample` is not a valid argument for this processor and will be ignored.
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Cat
